# EXP-010 (red-team): independent Merkle replay and negative controls

## 1 Requirements and predeclared hypothesis
H0: predecessor roots are reproducible with independent code; H1: at least one structural weakness exists in the duplicate-last-odd scheme. Python 3.12+, no network.

## 2 Input source, license, hashes
evidence/freeze/predecessor/* (metadata only, CC BY-NC source stays off-repo), evidence/redteam/independent_replay_magicstudiobox.json (receipt of the on-host run).

## 3 Frozen settings
scheme sha256(00||canonical_leaf); sha256(01||L||R); duplicate-last-odd; canonical JSON sort_keys, compact, ensure_ascii=False.

## 4 Deterministic preprocessing
none beyond canonical JSON.

## 5 Model/harness execution
none (integrity only).

## 6 Analysis
root recomputation, proof round trip, mutation matrix.

## 7 Negative controls
corrupt, reorder, remove, prefix swap, zero pad, duplicate-last collision.

## 8 Outputs
exp010_replay.json with SHA-256.

## 9 FCO/FCG dependency changes
none; adds a verified receipt leaf to the red-team successor.

## 10 Claim ceiling
Notebook PASS verifies the local replay only. It never upgrades NOT_TESTED, BLOCKED or NEGATIVE states, and it never attributes a causal gain to a hash, leaf or edge: hashes bind bytes, controlled interventions bind effects.


In [1]:
import json, hashlib, sys
from pathlib import Path
R = Path.cwd(); sys.path.insert(0, str(R / 'src')); sys.path.insert(0, str(R / 'scripts/redteam'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def jsonl(p): return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]
OUT = R / 'evidence/notebook_runs_redteam_v1/outputs'; OUT.mkdir(parents=True, exist_ok=True)
def emit(name, obj):
    p = OUT / name; p.write_text(json.dumps(obj, sort_keys=True, indent=2) + '\n'); print('OUTPUT', name, sha(p)); return p
from redteam_verify_sources import leaf, root, proof_ok, canon, sha_hex
rows = jsonl(R / 'evidence/freeze/predecessor/leaves.jsonl'); man = json.loads((R / 'evidence/freeze/predecessor/manifest.json').read_text())
leaves = [leaf(x) for x in rows]; got = root(leaves).hex()
assert got == man['root'] == '9e4d4df8aa7a887ecc8b792a1d099832b584c16f81ea821897fcd437f6fd3a9a'
assert len(rows) == man['leaf_count'] == 15
host = json.loads((R / 'evidence/redteam/independent_replay_magicstudiobox.json').read_text())
assert host['status']['root_728'] == 'PASS' and host['status']['root_15'] == 'PASS' and host['snapshot_728']['inclusion_proof_failures'] == 0
neg = {'corrupt': root(leaves[:3] + [leaf({'x': 1})] + leaves[4:]).hex() != got, 'reorder': root([leaves[1], leaves[0]] + leaves[2:]).hex() != got,
       'remove': root(leaves[:-1]).hex() != got, 'prefix_swap': root([leaf(x, b'\x01') for x in rows]).hex() != got,
       'zero_pad': root(leaves, dup_odd=False).hex() != got, 'duplicate_last_same_root': root(leaves + [leaves[-1]]).hex() == got}
assert all(neg.values())
res = {'root_15_recomputed': got, 'host_receipt_status': host['status'], 'negative_controls': neg,
       'finding_F01': 'duplicate-last-odd: appending a copy of the last leaf keeps the root; leaf_count must be bound outside the root (it is, unsigned)',
       'claim': 'INTEGRITY_ONLY_NOT_TRUTH'}
emit('exp010_replay.json', res); print(json.dumps(res, indent=2))

OUTPUT exp010_replay.json 472982b36a74e22c644d823bd4f675c8d2246b44e3a8b2e0738222667e304035
{
  "root_15_recomputed": "9e4d4df8aa7a887ecc8b792a1d099832b584c16f81ea821897fcd437f6fd3a9a",
  "host_receipt_status": {
    "duplicate_last_leaf_collision_present": true,
    "oracle_isolation_public_inputs": "PASS",
    "pair_binding_38": "PASS",
    "root_15": "PASS",
    "root_728": "PASS",
    "split_regeneration": "PASS"
  },
  "negative_controls": {
    "corrupt": true,
    "reorder": true,
    "remove": true,
    "prefix_swap": true,
    "zero_pad": true,
    "duplicate_last_same_root": true
  },
  "finding_F01": "duplicate-last-odd: appending a copy of the last leaf keeps the root; leaf_count must be bound outside the root (it is, unsigned)",
  "claim": "INTEGRITY_ONLY_NOT_TRUTH"
}
